# FishSizeEstimate — SFISHTRACK Supervised Segmentation & Held-out Evaluation (V11)

**Purpose:** replace the unreliable OpenCV foreground heuristic with a **trained fish instance-segmentation model**. This notebook consumes the Kaggle dataset at `/kaggle/input/datasets/wafabila/sfishtrack-dataset`, validates the converted YOLO labels, trains a YOLO segmentation baseline, evaluates on a video-disjoint held-out test split, and exports reproducible results and weights.

**Scientific boundary:** SFISHTRACK provides fish masks, **not calibrated fish length in centimetres**. Training a segmenter is necessary for identifying fish but does *not* solve stereo correspondence, fish head/tail verification or metric calibration. No MAE/RMSE in cm is claimed here. Existing stereo research remains a separate stage. The original COCO RLE masks were simplified to polygons; 4,174 source frames were excluded by local conversion, so the audit and visual QA are mandatory.

**Kaggle Settings:** GPU T4 (or P100), Internet ON for installation/model weights if necessary, then **Run All / Save Version → Save & Run All**. This notebook does not invent metrics: numbers appear only after successful execution.

## 0. Configuration, GPU and reproducibility

Adjust `EPOCHS`, `IMGSZ`, and `BATCH` only in this cell. Default trains on the **full available training split**; no random frame-level resplitting. Reduced-resolution/epoch runs are experiments, not final claims.

In [ ]:
from pathlib import Path
import os, sys, json, csv, math, random, time, platform, subprocess, shutil, warnings, hashlib, zipfile
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

REQUESTED_INPUT = Path('/kaggle/input/datasets/wafabila/sfishtrack-dataset')
OUTPUT = Path('/kaggle/working/fishsizeestimate_sfishtrack_v11')
OUTPUT.mkdir(parents=True, exist_ok=True)
EPOCHS = 30
IMGSZ = 512
BATCH = 16
WORKERS = 2
PATIENCE = 10
MODEL_NAME = 'yolo11n-seg.pt'       # small, established baseline; train entire available train split
RUN_SECOND_MODEL = False           # optional yolo11s-seg.pt comparison: much more compute
MODEL_2_NAME = 'yolo11s-seg.pt'
RUN_ONNX_EXPORT = False            # optional; install dependencies if this is enabled
MAX_QA_SAMPLES_PER_SPLIT = 4

import torch
print('Python:', sys.version.split()[0], 'Torch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
else:
    raise RuntimeError('GPU not enabled. Kaggle Settings > Accelerator > GPU; restart session and Run All.')
DEVICE = 0
(OUTPUT/'settings.json').write_text(json.dumps({
    'seed':SEED,'model':MODEL_NAME,'second_model':MODEL_2_NAME if RUN_SECOND_MODEL else None,
    'epochs':EPOCHS,'imgsz':IMGSZ,'batch':BATCH,'patience':PATIENCE,
    'input_path':str(REQUESTED_INPUT),'torch_version':torch.__version__,
    'gpu':torch.cuda.get_device_name(0)
},indent=2),encoding='utf-8')

## 1. Dependencies

No global upgrade of NumPy/Pillow is forced. Use the Kaggle environment's compatible PyTorch. `ultralytics` is installed only if missing. If Kaggle Internet is disabled and ultralytics/weights are not cached, enable Internet once.

In [ ]:
import importlib.util
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics>=8.3,<9'])
import ultralytics
from ultralytics import YOLO
import numpy as np
import cv2
import matplotlib.pyplot as plt
import yaml
from PIL import Image
print('Ultralytics:',ultralytics.__version__, 'NumPy:',np.__version__)

## 2. Discover the Kaggle data mount (do not assume directory layout)

Kaggle's actual mount sometimes differs from the user-facing dataset path. Search by **required `data.yaml`, `images/{train,val,test}`, and `labels/{train,val,test}`**, then print the resolved location.

In [ ]:
def valid_dataset_dir(d):
    return (d/'data.yaml').is_file() and all((d/'images'/s).is_dir() and (d/'labels'/s).is_dir() for s in ('train','val','test'))

candidate_roots = [REQUESTED_INPUT,Path('/kaggle/input/sfishtrack-dataset'),Path('/kaggle/input')]
candidates=[]
for base in candidate_roots:
    if not base.exists(): continue
    if valid_dataset_dir(base): candidates.append(base)
    for p in base.rglob('data.yaml'):
        if valid_dataset_dir(p.parent): candidates.append(p.parent)

candidates = sorted(set(candidates),key=lambda p:(0 if REQUESTED_INPUT in p.parents or p==REQUESTED_INPUT else 1,len(p.parts)))
if not candidates:
    raise FileNotFoundError('No YOLO dataset found: require data.yaml + images/train,val,test + labels/train,val,test. Check Add Input in Kaggle.')
DATA_ROOT = candidates[0]
print('Dataset root:',DATA_ROOT)
print('Available candidates:',[str(x) for x in candidates[:8]])
raw_yaml=yaml.safe_load((DATA_ROOT/'data.yaml').read_text(encoding='utf-8-sig'))
print('Uploaded YAML:',raw_yaml)
assert 'names' in raw_yaml, 'Missing class mapping'
classes = raw_yaml['names']
assert list(classes.values())==['fish'] if isinstance(classes,dict) else list(classes)==['fish'], f'Expected fish class, got {classes}'

## 3. Comprehensive dataset preflight — label structure, file correspondence, video-disjoint splits

Check *all* exported frame-label pairs and polygon coordinates, not just file counts. Crucially, verify video IDs are disjoint across train/validation/test. This detects data corruption and accidental evaluation leakage. Original source COCO annotations are not included in the exported YOLO ZIP, so this does not retroactively validate excluded RLE masks.

In [ ]:
from tqdm.auto import tqdm
splits = ('train','val','test')
image_ext = {'.jpg','.jpeg','.png','.webp','.bmp'}
images_by_split={}
labels_by_split={}
video_by_split={}
problems=[]
summary={}
for split in splits:
    im_dir = DATA_ROOT/'images'/split
    lb_dir = DATA_ROOT/'labels'/split
    images=sorted(p for p in im_dir.iterdir() if p.is_file() and p.suffix.lower() in image_ext)
    labels=sorted(p for p in lb_dir.glob('*.txt') if p.is_file())
    images_by_split[split]=images
    labels_by_split[split]=labels
    video_by_split[split]=set()
    label_ids={p.stem for p in labels}
    image_ids={p.stem for p in images}
    for stem in sorted(image_ids-label_ids)[:25]: problems.append(f'{split}: missing label {stem}')
    for stem in sorted(label_ids-image_ids)[:25]: problems.append(f'{split}: orphan label {stem}')
    videos=set()
    instance_count=0; empty_labels=0; polygons=0
    for im in tqdm(images,desc=f'Validate {split}'):
        prefix=im.stem.split('_',1)[0]
        if not prefix.startswith('video') or not prefix[5:].isdigit():
            problems.append(f'{split}: filename lacks video group {im.name}')
        videos.add(prefix)
        label=lb_dir/(im.stem+'.txt')
        if not label.exists():continue
        lines=[x.strip() for x in label.read_text(encoding='utf-8-sig').splitlines() if x.strip()]
        if not lines:empty_labels+=1
        for ln in lines:
            try:
                vals=np.fromstring(ln,sep=' ')
                if len(vals)<7 or len(vals)%2 != 1 or not np.isfinite(vals).all() or int(vals[0])!=0 or vals[0]!=0 or np.any(vals[1:] < 0) or np.any(vals[1:] > 1):
                    raise ValueError(f'invalid geometry ({len(vals)} tokens)')
            except Exception as ex:
                problems.append(f'{split}: invalid label {label.name}: {ex}')
            instance_count+=1;polygons+=1
    video_by_split[split]=videos
    summary[split]={'images':len(images),'labels':len(labels),'instances':instance_count,'empty_labels':empty_labels,'video_count':len(videos)}
for a,b in [('train','val'),('train','test'),('val','test')]:
    intersection=video_by_split[a]&video_by_split[b]
    if intersection: problems.append(f'LEAKAGE: shared videos {a}/{b}: {sorted(intersection)}')

print(json.dumps(summary,indent=2))
print('Problems count:',len(problems))
print('First problems:',problems[:25])
(OUTPUT/'dataset_preflight.json').write_text(json.dumps({
    'dataset_root':str(DATA_ROOT),'splits':summary,'video_groups':{k:sorted(v) for k,v in video_by_split.items()},
    'problems':problems[:1000]
},indent=2),encoding='utf-8')
assert all(summary[s]['images']>0 for s in splits),'Empty split'
assert not problems,'Dataset preflight failed; read dataset_preflight.json. Do not train with broken labels or leaking splits.'

## 4. Reconstruct polygon overlays for visual quality assurance

Sample real training/validation/test frames and draw colored polygons *on the corresponding image*. Check head, body, fins, tail, split masks, or mislabeled backgrounds. This is a **visual QA aid**, not an accuracy metric. You can stop here if overlays look wrong.

In [ ]:
from matplotlib.patches import Polygon
QA_DIR=OUTPUT/'qa_overlays';QA_DIR.mkdir(exist_ok=True)

def render_overlay(im_path, label_path, save_path):
    with Image.open(im_path) as im: rgb=np.asarray(im.convert('RGB'))
    h,w=rgb.shape[:2]
    fig,ax=plt.subplots(figsize=(11,7))
    ax.imshow(rgb); ax.set_title(im_path.name,fontsize=10);ax.axis('off')
    count=0
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():continue
        vals=np.fromstring(line,sep=' ')
        xy=vals[1:].reshape(-1,2)*np.array([w,h])
        ax.add_patch(Polygon(xy,closed=True,facecolor='lime',edgecolor='yellow',alpha=.3,linewidth=1.3))
        count+=1
    ax.text(.02,.03,f'Instances: {count}',transform=ax.transAxes,color='white',bbox={'facecolor':'black','alpha':.6})
    fig.savefig(save_path,dpi=105,bbox_inches='tight');plt.close(fig)
    return count

qa_manifest=[]
for split in splits:
    files=images_by_split[split]
    chosen=random.Random(SEED+len(split)).sample(files,min(MAX_QA_SAMPLES_PER_SPLIT,len(files)))
    for idx,im in enumerate(chosen):
        out=QA_DIR/f'{split}_{idx:02d}.png'
        n=render_overlay(im,DATA_ROOT/'labels'/split/(im.stem+'.txt'),out)
        qa_manifest.append({'split':split,'image':im.name,'instances':n,'overlay':out.name})
(OUTPUT/'qa_manifest.json').write_text(json.dumps(qa_manifest,indent=2),encoding='utf-8')
fig,axes=plt.subplots(3,4,figsize=(17,11));
for i,split in enumerate(splits):
    for j in range(4):
        ax=axes[i,j];p=QA_DIR/f'{split}_{j:02d}.png'
        if p.exists():ax.imshow(Image.open(p))
        ax.axis('off');ax.set_title(f'{split} {j+1}',fontsize=9)
plt.tight_layout();plt.show();plt.close(fig)
print('QA image files:',QA_DIR)

## 5. Reproducible Kaggle-aware YAML

`path:` must point to the actual Kaggle mount; the Windows exporter removed the hardcoded local path. Never move or resplit the test set. Write the resolved configuration into `/kaggle/working`.

In [ ]:
CONFIG_PATH=OUTPUT/'dataset_kaggle.yaml'
config={
    'path':str(DATA_ROOT),
    'train':'images/train',
    'val':'images/val',
    'test':'images/test',
    'names':{0:'fish'}
}
CONFIG_PATH.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
print(CONFIG_PATH.read_text())
if (DATA_ROOT/'conversion_report.json').exists():
    shutil.copy2(DATA_ROOT/'conversion_report.json',OUTPUT/'source_conversion_report.json')

## 6. Train **real supervised instance segmentation** — full train split

A YOLO segmentation model is trained using fish polygons and validated on the preexisting validation split. Checkpoints contain segmentation heads (unlike the older contrastive visual encoder). `best.pt` is selected using validation only. Training will take appreciable time on 19k frames; Kaggle sessions have runtime limits. `resume` is handled in the following cell if interrupted.

In [ ]:
TRAIN_DIR=OUTPUT/'training'
TRAIN_DIR.mkdir(parents=True,exist_ok=True)

def train_experiment(model_name,run_name):
    model=YOLO(model_name)
    result=model.train(
        data=str(CONFIG_PATH),
        task='segment',
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        seed=SEED,
        deterministic=True,
        device=DEVICE,
        workers=WORKERS,
        project=str(TRAIN_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        optimizer='auto',
        amp=True,
        plots=True,
        cache=False,
        close_mosaic=10,
        save=True,
        verbose=True
    )
    best=TRAIN_DIR/run_name/'weights'/'best.pt'
    if not best.exists():raise RuntimeError(f'No best checkpoint generated at {best}')
    print('Saved best:',best,'size MB:',round(best.stat().st_size/1e6,2))
    return best

BEST_A=train_experiment(MODEL_NAME,'baseline_yolo11n_seg')

## 7. Optional stronger architecture (not mandatory)

When enabled, train a YOLO11s segmenter on the *same* train/validation split and compare validation segmentation mAP. This can be significantly slower, so leave disabled unless the baseline is inadequate and the GPU budget allows it.

In [ ]:
BEST_B=None
if RUN_SECOND_MODEL:
    BEST_B=train_experiment(MODEL_2_NAME,'candidate_yolo11s_seg')
else:
    print('Optional stronger model disabled; baseline alone is a valid completed experiment.')

## 8. Compare **validation** metrics and choose champion

Test set is intentionally untouched here. Compare segmentation mAP50-95 on validation. A model with a higher box mAP is not necessarily a better segmenter.

In [ ]:
def measure_val(weights,split='val'):
    model=YOLO(str(weights))
    result=model.val(
        data=str(CONFIG_PATH),split=split,imgsz=IMGSZ,batch=BATCH,
        device=DEVICE,workers=WORKERS,plots=True,
        project=str(OUTPUT/'evaluations'),name=f'{weights.parent.parent.name}_{split}',exist_ok=True,
        verbose=False
    )
    return {'mask_map50_95':float(result.seg.map),
            'mask_map50':float(result.seg.map50),
            'box_map50_95':float(result.box.map),
            'box_map50':float(result.box.map50)}

comparison={str(BEST_A):measure_val(BEST_A,'val')}
if BEST_B:comparison[str(BEST_B)]=measure_val(BEST_B,'val')
CHAMPION_PATH=Path(max(comparison,key=lambda p:comparison[p]['mask_map50_95']))
(OUTPUT/'validation_comparison.json').write_text(json.dumps({
    'candidate_metrics':comparison,'champion':str(CHAMPION_PATH),
    'selection_criterion':'validation mask mAP50-95 only'
},indent=2),encoding='utf-8')
print('Validation comparison:',json.dumps(comparison,indent=2))
print('CHAMPION:',CHAMPION_PATH)

## 9. Held-out **test evaluation exactly once**, after champion selection

Reports measured mask and bounding-box mAP. Because the exported dataset excluded 4,174 frames with failed polygon conversion, performance is conditional on the retained frames; it does not establish reliability on all original footage, different cameras, or real-world length estimation.

In [ ]:
TEST_METRICS=measure_val(CHAMPION_PATH,'test')
final_report={
    'status':'evaluated', 'task':'fish instance segmentation (not physical measurement)',
    'source_dataset':'SFISHTRACK YOLO converted labels',
    'dataset_root':str(DATA_ROOT),'dataset_splits':summary,
    'selection':'validation mask mAP50-95',
    'champion_weights':str(CHAMPION_PATH),
    'validation_metrics':comparison[str(CHAMPION_PATH)],
    'heldout_test_metrics':TEST_METRICS,
    'limitations':[
        'YOLO polygon labels approximate original COCO RLE masks',
        '4,174 original frames excluded during local export due to conversion failures',
        'Video-disjoint split; not verified fish-identity-disjoint across videos',
        'No fish lengths in centimetres are predicted or evaluated',
        'Results may not transfer to original stereo footage without domain adaptation'
    ]
}
(OUTPUT/'heldout_test_report.json').write_text(json.dumps(final_report,indent=2),encoding='utf-8')
print(json.dumps(final_report,indent=2))

## 10. Real predictions on a few held-out test frames

Visualize model segmentation overlays on **unseen test images**, not arbitrary gradients or OpenCV contour heuristics. These are qualitative examples; use test metrics for numerical evidence.

In [ ]:
CHAMPION=YOLO(str(CHAMPION_PATH))
PRED_DIR=OUTPUT/'predictions';PRED_DIR.mkdir(exist_ok=True)
test_examples=random.Random(SEED+991).sample(images_by_split['test'],min(12,len(images_by_split['test'])))
visual_manifest=[]
for idx,im_path in enumerate(test_examples):
    result=CHAMPION.predict(source=str(im_path),imgsz=IMGSZ,conf=0.25,device=DEVICE,verbose=False)[0]
    overlay=result.plot() # BGR
    save_path=PRED_DIR/f'test_{idx:02d}_{im_path.stem}.jpg'
    cv2.imwrite(str(save_path),overlay)
    num_detected=int(len(result.boxes)) if result.boxes is not None else 0
    num_masks=int(len(result.masks)) if result.masks is not None else 0
    visual_manifest.append({'image':im_path.name,'detected_instances':num_detected,'predicted_masks':num_masks,'output':save_path.name})
(OUTPUT/'prediction_manifest.json').write_text(json.dumps(visual_manifest,indent=2),encoding='utf-8')
fig,axes=plt.subplots(3,4,figsize=(18,12))
for ax,p in zip(axes.flat,sorted(PRED_DIR.glob('*.jpg'))):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB));ax.axis('off')
for ax in list(axes.flat)[len(test_examples):]:ax.axis('off')
plt.tight_layout();plt.show();plt.close(fig)
print('Prediction overlays:',PRED_DIR)

## 11. Export deployable model, manifest and API guidance

Save `best_fish_segmenter.pt` and `model_manifest.json`. Optional ONNX export is guarded because Kaggle package availability varies. The FastAPI backend must **load these weights and call `predict()`**; the existing OpenCV contour endpoint does not become an ML detector by renaming it.

In [ ]:
MODELS_DIR=OUTPUT/'models';MODELS_DIR.mkdir(exist_ok=True)
DEPLOY_PT=MODELS_DIR/'best_fish_segmenter.pt'
shutil.copy2(CHAMPION_PATH,DEPLOY_PT)

def sha256file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

model_manifest={
    'task':'fish instance segmentation','framework':'ultralytics',
    'ultralytics_version':ultralytics.__version__,
    'model_file':DEPLOY_PT.name,'sha256':sha256file(DEPLOY_PT),
    'classes':['fish'],'input_imgsz':IMGSZ,'confidence_example':0.25,
    'validation_mask_map50_95':comparison[str(CHAMPION_PATH)]['mask_map50_95'],
    'heldout_test_mask_map50_95':TEST_METRICS['mask_map50_95'],
    'does_predict_physical_length_cm':False,
    'provenance':'trained by this notebook from SFISHTRACK converted segmentation annotations'
}
(MODELS_DIR/'model_manifest.json').write_text(json.dumps(model_manifest,indent=2),encoding='utf-8')
if RUN_ONNX_EXPORT:
    exported=CHAMPION.export(format='onnx',imgsz=IMGSZ,opset=17,simplify=False,device='cpu')
    exported=Path(exported)
    if exported.is_file():shutil.copy2(exported,MODELS_DIR/'best_fish_segmenter.onnx')
print('Deployable weights:',DEPLOY_PT,round(DEPLOY_PT.stat().st_size/1e6,2),'MB')
print('Manifest:',MODELS_DIR/'model_manifest.json')

## 12. Capture environment, create output ZIP, verify integrity

The archive includes **actual checkpoint weights, QA overlays, predicted test examples, config, training plots and metric reports**. Original 22.45 GiB Kaggle input images are *not* re-zipped. ZIP CRC validation is mandatory and a missing model or metric report fails the cell.

In [ ]:
try:
    env=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True,timeout=60)
    (OUTPUT/'environment_pip_freeze.txt').write_text(env,encoding='utf-8')
except Exception as ex:print('Could not capture pip freeze:',ex)

readme = """# FishSizeEstimate SFISHTRACK V11 — trained segmentation artifacts

This artifact contains a trained fish instance-segmentation model and recorded held-out evaluation.
It does not estimate length in centimetres. Stereo metric reconstruction and independently verified fish measurements remain separate work.

## Model deployment
Install a compatible `ultralytics` version. Use `YOLO("models/best_fish_segmenter.pt")` and `model.predict(image, conf=0.25)`.
Inspect `models/model_manifest.json` and `heldout_test_report.json` for exact measured results.
"""
(OUTPUT/'README_ARTIFACTS.md').write_text(readme,encoding='utf-8')

assert DEPLOY_PT.is_file() and DEPLOY_PT.stat().st_size>100_000
assert (OUTPUT/'heldout_test_report.json').is_file()
assert len(list(PRED_DIR.glob('*.jpg'))) >= 1

ZIP=Path('/kaggle/working/FishSizeEstimate_SFISHTRACK_V11_Results.zip')
all_files=sorted(f for f in OUTPUT.rglob('*') if f.is_file())
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2,allowZip64=True) as z:
    for file in tqdm(all_files,desc='Packaging results'):
        z.write(file,arcname=file.relative_to(OUTPUT).as_posix())
with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()
    assert bad is None,f'ZIP CRC error in {bad}'
    assert 'models/best_fish_segmenter.pt' in z.namelist()
    assert 'heldout_test_report.json' in z.namelist()
print('SUCCESS:',ZIP,'MB:',round(ZIP.stat().st_size/1e6,1),'files:',len(all_files))
print('Trained checkpoint:',DEPLOY_PT)
print('Held-out mask mAP50-95:',TEST_METRICS['mask_map50_95'])

## 13. Next research stage — real fish length in cm (NOT yet solved)

To estimate physical length, the project needs verified stereo left/right frame pairing, calibrated/rectified camera geometry (including baseline units), reliable fish segmentation and head/tail correspondence in both views, a defensible 3D endpoint distance, and independent ground-truth fish lengths for MAE/RMSE reporting. **Do not derive centimetres from a single image's mask width.**

**Proposed downstream work:** integrate the `best_fish_segmenter.pt` weights into the FastAPI endpoint; measure segmentation quality on original stereo camera images; then continue the separate stereo-measurement investigation notebook with physically verified calibrations. A deployment demo must label physical length as *not measured* until evaluation is successful.